# Hierarchical process (with Tools call)

Agents orchestration with a manager agent.

# Installation

In [15]:
!pip install -q crewai crewai_tools

# Import Dependencies

In [16]:
import crewai
import crewai_tools

print(crewai.__version__)
print(crewai_tools.__version__)

1.15.10
1.15.10


In [10]:
# Import dependencies
import os
from crewai import Agent, Task, Crew, LLM, Process
from crewai_tools import SerperDevTool, DirectoryReadTool

# If loading from a .env file
# from dotenv import load_dotenv
# load_dotenv()


# Set API Keys

In [3]:
from google.colab import userdata
import os
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY_NEW')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')


# Create LLM Objects

In [ ]:
# -----------
# Create LLM
# -----------

# Create an LLM with a temperature of 0 to ensure deterministic outputs
# OPENAI LLMs
manager_llm = LLM(
         # model="gpt-5.4-mini",
          model="gpt-5.4-nano",
          base_url="https://api.openai.com/v1",
          api_key = os.environ["OPENAI_API_KEY"],
          temperature=0.2)

# GROQ hosted LLMs
llm = LLM(
     model="llama-3.3-70b-versatile",
     base_url="https://api.groq.com/openai/v1",
     api_key=os.environ["GROQ_API_KEY"],
     temperature=0.7)


# Tools

In [6]:
#------------------------------------------------------------------------
# Instantiate the Tools
#------------------------------------------------------------------------

search_tool = SerperDevTool()

docs_tool = DirectoryReadTool(directory='./blog-posts')

# Agents

In [11]:
# --------------
# Define Agents
# --------------
manager_agent = Agent(
    role="Content Manager",
    goal="Coordinate the blog creation process by assigning subtasks to the research specialist and content writer agents.",
    backstory=(
        "You are an experienced content manager. "
        "You decide the order of work, delegate to other coworker agents, and ensure the final task completion."
        "You do NOT solve the problem on your own. You need to coordinate amongst the coworker agents to get the task completed."
        "Ideally you should kick off the researcher first to collect information and then pass the research findings to the content writer agent."
        "Coworker agents may have access to specific tools that they should use to complete their tasks."
    ),
    llm=manager_llm, # Make sure to use a more capable LLM here
    max_iter=3,
    allow_delegation=True,
    verbose=True
)

research_agent = Agent(
    role="Research Specialist",
    goal="Gather accurate and relevant information in real-time for a given topic.",
    backstory="You are an expert in web research, skilled at finding key facts, statistics, and trends.",
    llm=llm,
    max_iter=3,
    tools=[search_tool],
    allow_delegation=False,
    verbose=True
)

writer_agent = Agent(
    role="Content Writer",
    goal="Produce high-quality blog articles from the provided research material.",
    backstory="You write clear, engaging, and well-structured content.",
    llm=llm,
    tools=[docs_tool],
    max_iter=3,
    max_rpm=15,
    allow_delegation=False,
    verbose=True
)


# Tasks

In [12]:
# --------------------
# Define Tasks
# --------------------
research_task = Task(
    description=(
        "Research the topic '{topic}'. Provide 5-7 bullet points "
        "with key facts, statistics, and relevant examples."
        "Use the provided tool to search for the latest information about the topic. "
        "Do NOT rely on memory — you must call the search tool at least once."
    ),
    expected_output="A bullet-point list of factual research notes.",
    agent=research_agent,
    verbose=True

)

writing_task = Task(
    description=(
        "Write a 500-word blog post on '{topic}' using this research done by the researcher."
    ),
    expected_output="A fully written blog post.",
    agent=writer_agent,

    # Let the manager pass the context as required.
    # You do not need to set any order of execution of the agents/tasks or pass the context
    # DO NOT SET THE FOLLOWING ATTRIBUTES
    # depends_on=[research_task],
    # context = [research_task], # This passes the context of the research task as a Task object not as strings as expected by the writin agent
                                 # Comment this line and just use depends_on to pass the context as a string

    output_file="blog-posts/new_tech_post_{topic}.md",  # The final blog post will be saved here
    verbose=True
)


# Crew (Orchestration Layer)

In [13]:
# -------------------------------------
# Define Crew with Hierarchical Process
# -------------------------------------
crew = Crew(
    agents=[research_agent, writer_agent], # We do NOT specify the manager agent in this list
    tasks=[research_task, writing_task],   # Tasks to be delegated and executed under the manager's supervision
    process=Process.hierarchical,    # <--- THIS sets the hierarchical workflow
    manager_agent=manager_agent,     # <--- EITHER Explicitly set the manager agent
    # manager_llm="openai/gpt-4o", # <--- OR Explicitly set the manager LLM, make sure to use a more capable LLM here
    planning=True,
    verbose=True
)


In [17]:
# -------------
# Run the Crew
# -------------
# result = await crew.kickoff_async(inputs={"topic": "The future of job market for CS undergrad in the era of Gen AI"})
result = await crew.kickoff_async(inputs={"topic": "The future of Humanity as we tend to achieve AGI"})
# result = await crew.kickoff_async(inputs={"topic": "About OpenAI's latest GPT-5 model. Major technical heighlights as 5 bullet points. Additional 3 bullet points highlighting how it is better than GPT-4"})

print("\n=== FINAL OUTPUT ===")
print(result)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5bd5c71d-a6af-494e-a7a3-18baf261d5aa                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[2026-08-03 05:10:12][INFO]: Planning the crew execution


╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Based on these tasks summary:                                                                            │
│                  Task Number 1 - Research the topic 'The future of Humanity as we tend to achieve AGI'.         │
│  Provide 5-7 bullet points with key facts, statistics, and relevant examples.Use the provided tool to search    │
│  for the latest information about the topic. Do NOT rely on memory — you must call the search tool at least     │
│  once.                                                                                                          │
│                  "task_description": Research the topic 'The future of Humanity as we tend to achieve AGI'.     │
│  Provide 5-7 bullet points with key facts, statistics, and relevant examples.Use the provided tool to search    │
│  for the latest information about the topic. Do NOT rely on memory — you must call the search tool at least     │
│  once.                                                                                                          │
│                  "task_expected_output": A bullet-point list of factual research notes.                         │
│                  "agent": Content Manager                                                                       │
│                  "agent_goal": Coordinate the blog creation process by assigning subtasks to the research       │
│  specialist and content writer agents.                                                                          │
│                  "task_tools": [SerperDevTool(name='Search the internet with Serper', description="A tool that  │
│  can be used to search the internet with a search_query. Supports different search types: 'search' (default),   │
│  'news'", env_vars=[EnvVar(name='SERPER_API_KEY', description='API key for Serper', required=True,              │
│  default=None)], args_schema=<class 'crewai_tools.tools.serper_dev_tool.serper_dev_tool.SerperDevToolSchema'>,  │
│  result_schema=None, description_updated=False, cache_function=<function _default_cache_function at             │
│  0x783aa861aac0>, result_as_answer=False, max_usage_count=None, tool_failure_policy=None,                       │
│  current_usage_count=0, base_url='https://google.serper.dev', n_results=10, save_file=False,                    │
│  search_type='search', country='', location='', locale='',                                                      │
│  tool_type='crewai_tools.tools.serper_dev_tool.serper_dev_tool.SerperDevTool')]                                 │
│                  "agent_tools": "agent has no tools"                                                            │
│                  Task Number 2 - Write a 500-word blog post on 'The future of Humanity as we tend to achieve    │
│  AGI' using this research done by the researcher.                                                               │
│                  "task_description": Write a 500-word blog post on 'The future of Humanity as we tend to        │
│  achieve AGI' using this research done by the researcher.                                                       │
│                  "task_expected_output": A fully written blog post.                                             │
│                  "agent": Content Manager                                                                       │
│                  "agent_goal": Coordinate the blog creation process by assigning subtasks to the research       │
│  specialist and content writer agents.                                                                          │
│                  "task_tools": [DirectoryReadTool(name=

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Based on these tasks summary:                                                                            │
│                  Task Number 1 - Research the topic 'The future of Humanity as we tend to achieve AGI'.         │
│  Provide 5-7 bullet points with key facts, statistics, and relevant examples.Use the provided tool to search    │
│  for the latest information about the topic. Do NOT rely on memory — you must call the search tool at least     │
│  once.                                                                                                          │
│                  "task_description": Research the topic 'The future of Humanity as we tend to achieve AGI'.     │
│  Provide 5-7 bullet points with key facts, statistics, and relevant examples.Use the provided tool to search    │
│  for the latest information about the topic. Do NOT rely on memory — you must call the search tool at least     │
│  once.                                                                                                          │
│                  "task_expected_output": A bullet-point list of factual research notes.                         │
│                  "agent": Content Manager                                                                       │
│                  "agent_goal": Coordinate the blog creation process by assigning subtasks to the research       │
│  specialist and content writer agents.                                                                          │
│                  "task_tools": [SerperDevTool(name='Search the internet with Serper', description="A tool that  │
│  can be used to search the internet with a search_query. Supports different search types: 'search' (default),   │
│  'news'", env_vars=[EnvVar(name='SERPER_API_KEY', description='API key for Serper', required=True,              │
│  default=None)], args_schema=<class 'crewai_tools.tools.serper_dev_tool.serper_dev_tool.SerperDevToolSchema'>,  │
│  result_schema=None, description_updated=False, cache_function=<function _default_cache_function at             │
│  0x783aa861aac0>, result_as_answer=False, max_usage_count=None, tool_failure_policy=None,                       │
│  current_usage_count=0, base_url='https://google.serper.dev', n_results=10, save_file=False,                    │
│  search_type='search', country='', location='', locale='',                                                      │
│  tool_type='crewai_tools.tools.serper_dev_tool.serper_dev_tool.SerperDevTool')]                                 │
│                  "agent_tools": "agent has no tools"                                                            │
│                  Task Number 2 - Write a 500-word blog post on 'The future of Humanity as we tend to achieve    │
│  AGI' using this research done by the researcher.                                                               │
│                  "task_description": Write a 500-word blog post on 'The future of Humanity as we tend to        │
│  achieve AGI' using this research done by the researcher.                                                       │
│                  "task_expected_output": A fully written blog post.                                             │
│                  "agent": Content Manager                                                                       │
│                  "agent_goal": Coordinate the blog creation process by assigning subtasks to the research       │
│  specialist and content writer agents.                                                                          │
│                  "task_tools": [DirectoryReadTool(name=

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the topic 'The future of Humanity as we tend to achieve AGI'. Provide 5-7 bullet points with    │
│  key facts, statistics, and relevant examples.Use the provided tool to search for the latest information about  │
│  the topic. Do NOT rely on memory — you must call the search tool at least once.1. Start by clearly defining    │
│  the research scope: the future of humanity as AGI is approached, focusing on social, economic, safety,         │
│  governance, labor, and scientific implications.                                                                │
│  2. Use the SerperDevTool search tool at least once, and preferably multiple times, to gather the latest        │
│  authoritative information. Search with a mix of broad and targeted queries such as: "AGI future humanity       │
│  latest research", "AGI risk statistics 2024", "AGI governance policy updates", "AI labor market impact         │
│  latest", and "frontier AI safety examples".                                                                    │
│  3. Prioritize recent and credible sources in the search results, including major research organizations,       │
│  policy institutions, reputable news outlets, and technical labs. Filter out opinion-only or non-factual        │
│  sources unless they provide cited data.                                                                        │
│  4. Collect 5-7 strong factual notes that each include one key fact, statistic, or concrete example. Ensure     │
│  the notes cover a balanced view: transformative benefits, labor displacement, alignment/safety risk, economic  │
│  concentration, governance responses, and examples of real-world AGI-adjacent progress.                         │
│  5. For each bullet point, extract the most relevant data available from the search results, including dates,   │
│  percentages, quoted projections, named organizations, or specific incidents/examples. Verify that each bullet  │
│  is grounded in a source rather than inference.                                                                 │
│  6. If search results contain conflicting claims, prefer the most recent and most widely corroborated           │
│  information. Note the consensus or uncertainty carefully rather than overstating conclusions.                  │
│  7. Structure the final output strictly as a bullet-point list of factual research notes, concise but           │
│  information-rich, with 5-7 bullets total. Make sure each bullet is independently useful for the later          │
│  blog-writing task.                                                                                             │
│  ID: dd1fbab2-3a2a-4dfd-adb4-018e7baf546d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Maximum iterations reached. Requesting final answer.
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Manager                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # The future of Humanity as we tend to achieve AGI                                                             │
│                                                                                                                 │
│  Imagine a future where machines don’t just assist with tasks, but increasingly understand goals, reason        │
│  across domains, and help coordinate complex decisions. That’s the promise many people associate with           │
│  **AGI**—and it’s also why the question “What happens to humanity as we get there?” can’t be answered with      │
│  hype alone.                                                                                                    │
│                                                                                                                 │
│  AGI progress is often described as a gradual, uneven climb. Benchmark-style capability gains show that         │
│  frontier systems are improving on some dimensions faster than others, with one 2024 comparison reporting       │
│  **GPT-4o at 69.1%** versus **Gemini 1.5 Pro and Claude 3 Opus at ~58.5%** on the cited measures. The key       │
│  takeaway isn’t that any single model equals “AGI” today—it’s that the path toward it may look like steady      │
│  advances that accumulate across skills.                                                                        │
│                                                                                                                 │
│  ## 1) Governance is shifting from promises to enforceable rules                                                │
│                                                                                                                 │
│  As capabilities grow, governments are moving from broad “principles” toward **rules that can be enforced**. A  │
│  major signal is the **EU AI Act**, which **entered into force on 1 Aug 2024** and is scheduled to **apply on   │
│  2 Aug 2026** (with phased obligations). The Act includes **prohibited AI practices** and requirements for      │
│  **high-risk systems**, showing that policymakers are preparing for advanced capabilities with compliance       │
│  mechanisms—not relying solely on voluntary guidance.                                                           │
│                                                                                                                 │
│  This matters for humanity’s future because AGI-like systems would likely affect critical areas—health,         │
│  education, employment, and public services. When governance becomes operational, it can shape not only what    │
│  gets built, but how safely and transparently it’s deployed.                                                    │
│                                                                                                                 │
│  ## 2) The upside is real—but distributional effects are central                                                │
│                                                                                                                 │
│  The economic and societal upside is a major expectation. The **World Economic Forum’s “Preparing for           │
│  Artificial General Intelligence 2025”** frames AGI as potentially driving major advances in **economic         │
│  growth, healthcare outcomes, and climate solutions**. 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a 500-word blog post on 'The future of Humanity as we tend to achieve AGI' using this research     │
│  done by the researcher.1. First, read and inspect the contents of the ./blog-posts directory using the         │
│  DirectoryReadTool to identify any existing files, draft structures, style references, or required output       │
│  naming conventions.                                                                                            │
│  2. Review the researcher’s bullet-point notes carefully and extract the central themes that should drive the   │
│  blog post: what AGI is expected to change, what risks and opportunities exist, and what examples best support  │
│  the argument.                                                                                                  │
│  3. Plan a clear 500-word blog structure before drafting: introduction, 2-3 body sections, and conclusion.      │
│  Keep the piece coherent, reader-friendly, and centered on the future of humanity as AGI is approached.         │
│  4. Use the research notes as the factual backbone. Convert the bullet points into a compelling narrative       │
│  without adding unsupported claims. Keep the tone informative, balanced, and accessible to a general audience.  │
│  5. Ensure the article includes a strong hook, clear transitions, and a thoughtful conclusion that reflects     │
│  both the promise and the responsibility of AGI development.                                                    │
│  6. Target approximately 500 words, staying close enough to the requirement while preserving readability.       │
│  Avoid filler and avoid repeating the same point in multiple ways.                                              │
│  7. Before finalizing, compare the draft against any relevant files discovered in ./blog-posts to align         │
│  formatting, voice, or content expectations if such examples exist.                                             │
│  8. Deliver the final output as a fully written blog post, polished for clarity, grammar, and flow, and make    │
│  sure it remains faithful to the researcher’s evidence.                                                         │
│  Agent: Content Manager                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== FINAL OUTPUT ===
# The future of Humanity as we tend to achieve AGI

Imagine a future where machines don’t just assist with tasks, but increasingly understand goals, reason across domains, and help coordinate complex decisions. That’s the promise many people associate with **AGI**—and it’s also why the question “What happens to humanity as we get there?” can’t be answered with hype alone.

AGI progress is often described as a gradual, uneven climb. Benchmark-style capability gains show that frontier systems are improving on some dimensions faster than others, with one 2024 comparison reporting **GPT-4o at 69.1%** versus **Gemini 1.5 Pro and Claude 3 Opus at ~58.5%** on the cited measures. The key takeaway isn’t that any single model equals “AGI” today—it’s that the path toward it may look like steady advances that accumulate across skills.

## 1) Governance is shifting from promises to enforceable rules

As capabilities grow, governments are moving from broad “principles” toward *

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯